# Sales rep measures: Box-Cox transformation as a warning

**Data Transformation** · a worked example from the StatNex teaching guide

**The story.** A sales manager has a quarter's activity figures for 20 sales reps: calls, meetings, emails, deals closed, revenue and customer rating. The measures are on very different scales.

**The question.** How can you compare a rep's strengths across measures that use different units?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `12_utility/data_transformation.csv` (20 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Name,Calls per day,Meetings per week,Emails per day,Deals closed,Revenue ($000),Customer rating
Alex,53.0,8.0,45.0,16.0,108.6,4.3
Bea,41.0,7.0,32.0,10.0,66.3,4.1
Chen,54.0,13.0,34.0,19.0,129.7,4.6
Dana,40.0,8.0,12.0,11.0,64.3,4.2
Eli,48.0,10.0,48.0,12.0,87.4,4.3
Farah,48.0,9.0,51.0,16.0,108.8,4.1
Gus,38.0,8.0,72.0,8.0,41.7,3.7
Hana,38.0,8.0,54.0,13.0,72.7,4.7
Ivan,38.0,10.0,46.0,14.0,98.6,3.7
Jo,38.0,7.0,13.0,11.0,67.2,4.1
Kim,37.0,7.0,49.0,8.0,58.2,4.2
Leo,43.0,10.0,40.0,14.0,91.0,4.0
Mona,46.0,7.0,24.0,14.0,81.2,4.5
Nate,28.0,12.0,39.0,13.0,85.6,4.6
Omar,45.0,12.0,50.0,19.0,109.3,4.5
Pia,56.0,12.0,63.0,21.0,131.6,3.8
Quinn,39.0,8.0,24.0,12.0,72.2,4.2
Raj,43.0,7.0,18.0,14.0,100.1,4.2
Sofia,34.0,11.0,49.0,18.0,101.5,3.9
Tom,53.0,12.0,41.0,19.0,119.2,4.4

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# Each variable is transformed on its own; text and blank cells are left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Calls per day", "Meetings per week", "Emails per day", "Deals closed", "Revenue ($000)", "Customer rating"]
numeric = data[variables].apply(pd.to_numeric, errors="coerce")
columns = {name: numeric[name].dropna() for name in variables}

# Variance and SD with denominator n - 1 (the data are a sample).
ddof = 1


def describe(x):
    return [len(x), x.mean(), x.var(ddof=ddof), x.std(ddof=ddof)]

## Before the transformation

In [ ]:
before = pd.DataFrame({name: describe(x.to_numpy()) for name, x in columns.items()},
                      index=["n", "mean", "variance", "SD"])
print(before)

## The transformation

In [ ]:
# Box-Cox with lambda = a: (x^a - 1) / a, or log x when a = 0. Positive x only.
# lambda below 1 pulls in a long right tail (1/2 is close to a square root, 0 is the
# log); lambda = 1 only shifts the values. Here lambda is fixed by you; leaving out
# lmbda makes scipy.stats.boxcox estimate it from the data by maximum likelihood.
lam = float("0.5")


def transform(x):
    return stats.boxcox(x, lmbda=lam)


# The whole column is transformed at once: a mean- or rank-based conversion has to
# see every case. Missing cells stay missing, in their original rows.
transformed = {name: pd.Series(transform(x.to_numpy()), index=x.index)
               for name, x in columns.items()}
result = pd.DataFrame(transformed).reindex(data.index)

## After the transformation

In [ ]:
after = pd.DataFrame({name: describe(x.to_numpy()) for name, x in transformed.items()},
                     index=["n", "mean", "variance", "SD"])
print()
print(after)

## Shape of the distribution

In [ ]:
# Skewness (asymmetry: 0 for a symmetric shape, positive for a long right tail) and
# excess kurtosis (tail weight: 0 for a normal distribution), before and after.
# bias=False gives the small-sample-corrected G1 and G2 that SAS, SPSS and Excel's
# SKEW and KURT print. With n = 20 the skewness of a sample from a perfectly
# symmetric population still wanders by about +/-0.5 (its standard error is about
# sqrt(6 / n)), so only values well beyond that signal a real skew. A transformation
# that only shifts and rescales, like the z score, leaves both unchanged.
shape = pd.DataFrame(
    {name: [stats.skew(columns[name], bias=False), stats.kurtosis(columns[name], bias=False),
            stats.skew(transformed[name], bias=False),
            stats.kurtosis(transformed[name], bias=False)]
     for name in variables},
    index=["skewness before", "kurtosis before", "skewness after", "kurtosis after"],
)
print()
print(shape)

## Transformed data

In [ ]:
# In the original row order, so it can be pasted back beside the data.
result.to_csv("transformed.csv", index=False)
print()
print(result)

## What the result says

The Box-Cox transformation, here with parameter a = 0.5, bends the scale of a measure. It is meant for skewed data, where a few very large values pull one side out. Skewness measures this: 0 means a balanced shape.

These measures were never skewed. Before the transformation every skewness is small, for example -0.17 for emails and -0.03 for revenue. After it, emails has -0.64 and revenue -0.33, so both are more lopsided than before. The measures are also still on different scales: the mean is 11.07 for calls and 2.10 for rating.

So this run is a warning. Box-Cox does not help to compare reps across measures, and on balanced data it can do harm. Use z scores for that question.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.